<a href="https://colab.research.google.com/github/JF11579/PSID_For_the_Rest_of_US/blob/main/Three_Generation_Family_Explorer_v3_CORRECTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌳 Three-Generation PSID Family Tree Explorer

**Longitudinal Study: 1968-Present**

Explore how **1968 homeownership** affected educational outcomes measured decades later:

- **Generation 1 (Blue/Purple)**: Household heads in 1968 - Homeownership status
- **Generation 2 (Colored)**: Their children - Education measured as adults
- **Generation 3 (Colored)**: Grandchildren - Education measured as adults (1990s-2020s)

**Color = Education Level** | **Shape = Homeownership**

This shows the **long-term intergenerational effects** of homeownership!

---

### Data Requirements

This notebook builds its three-generation dataset from **two files** produced by the earlier notebooks in this project:

1. **`psid_intergenerational_clean.csv`** — the analytic dataset from Notebook 01.2 (Prepare Dataset). Each row is a parent–child pair linking a 1968 household head (Generation 1) to one of their children (Generation 2), with recoded variables for homeownership, education, income, race, and more.

2. **`fim15712_gidpro_BO_2_BAL.csv`** — the PSID FIMS genealogy file, which maps parent–child relationships using 1968 family interview IDs. We chain this file to extend the two-generation links from Notebook 01 into three generations: G1 → G2 → G3.

Both files live in the project data/outputs folders on Google Drive. If you've already run Notebook 01.2, these files are ready to go.

### How the Bridge Works

Notebook 01.2 gives us G1–G2 pairs. To get G3, we look at which G2 individuals are *themselves* listed as parents in the FIMS file. Their children become G3. We then merge G3's outcome data (education, homeownership) from the PSID extract to build the full three-generation tree.

## A Note on Sample Size

The three-generation sample is intentionally small. To appear in this dataset, a family must clear several hurdles: the G1 household head must be in the 1968 PSID sample with homeownership data; their G2 child must also appear in FIMS with education and homeownership outcomes; and that G2 child must themselves be a parent in FIMS whose G3 children are old enough (25+) to have completed their education and entered the housing market by 2023.
That last requirement is the biggest filter. Many G3 grandchildren are simply too young — born after 1998, they haven't yet reached the age where adult outcomes like homeownership can be meaningfully measured. As the PSID continues to follow these families, this three-generation sample will grow substantially.
For now, treat the family trees in this notebook as illustrative — real families showing how homeownership, education, and opportunity flow (or don't) across generations. The statistical analysis in Notebook 02 is where the larger two-generation sample does the heavy inferential lifting.

In [ ]:
# Install required packages
!pip install networkx matplotlib ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.1 MB/s eta 0:00:00


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from ipywidgets import interact, widgets, HBox, VBox, Button, Output
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

Mounted at /content/drive
✅ Google Drive mounted!


## 1. Load Data and Build Three-Generation Links

We load the two source files and construct the three-generation dataset.

**Step 1:** Load `psid_intergenerational_clean.csv` from Notebook 01.2. This gives us G1 → G2 pairs with all recoded variables.

**Step 2:** Load the FIMS genealogy file and build person IDs. The FIMS file maps `G1ID68 + G1PN` (parent) to `G2ID68 + G2PN` (child) using the standard PSID person_id construction: `(ID68 × 1000) + PN`.

**Step 3:** Chain the linkage. For each G2 person in our Notebook 01 data, check if they appear as a parent in FIMS. If so, their FIMS children become G3.

In [ ]:
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
# These must match the paths used in Notebook 01.2
BASE = '/content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2'
DATA_DIR   = os.path.join(BASE, 'data')
OUTPUT_DIR = os.path.join(BASE, 'outputs')

# File produced by Notebook 01.2 — the two-generation analytic dataset
NB01_OUTPUT = os.path.join(OUTPUT_DIR, 'psid_intergenerational_clean.csv')

# FIMS genealogy file — maps parent–child links using 1968 family IDs
FIMS_FILE  = os.path.join(DATA_DIR, 'fim15712_gidpro_BO_2_BAL.csv')

# The original PSID extract — needed for G3 outcome variables
PSID_FILE  = os.path.join(DATA_DIR, 'J358642.csv')

# Verify files exist
for label, path in [('Notebook 01 output', NB01_OUTPUT), ('FIMS file', FIMS_FILE), ('PSID extract', PSID_FILE)]:
    if os.path.exists(path):
        print(f'✅ {label}: {path}')
    else:
        print(f'❌ {label} NOT FOUND: {path}')

✅ Notebook 01 output: /content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2/outputs/psid_intergenerational_clean.csv
✅ FIMS file: /content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2/data/fim15712_gidpro_BO_2_BAL.csv
✅ PSID extract: /content/drive/MyDrive/Colab Notebooks/Projects/PSID/PSID_2026/PSID_book_v2/data/J358642.csv


### Build the Three-Generation Dataset

The key insight: in Notebook 01.2's output, `parent_id` is the G1 (1968 household head) and `child_id` is G2 (their child). To find G3, we look for G2 individuals who appear as parents in the FIMS genealogy. Their children are G3.

We then merge G3's own outcome variables (education, homeownership) from the PSID extract, applying the same recoding rules used in Notebook 01.2.

In [ ]:
# ── Step 1: Load the two-generation data from Notebook 01.2 ──────────────────
print('📂 Loading Notebook 01.2 output (G1–G2 pairs)...')
df_nb01 = pd.read_csv(NB01_OUTPUT)
print(f'   Loaded {len(df_nb01):,} rows')
print(f'   Columns: {list(df_nb01.columns)}')

# ── Step 2: Load FIMS and build linkage ──────────────────────────────────────
print('\n📂 Loading FIMS genealogy file...')
fims_raw = pd.read_csv(FIMS_FILE)
fims = fims_raw.copy()
fims['fims_parent_id'] = (fims['G1ID68'] * 1000) + fims['G1PN']
fims['fims_child_id']  = (fims['G2ID68'] * 1000) + fims['G2PN']
fims = fims[['fims_parent_id', 'fims_child_id']].drop_duplicates()
print(f'   FIMS linkage pairs: {len(fims):,}')

# ── Step 3: Chain to get G3 ──────────────────────────────────────────────────
# G2 people from Notebook 01 are in the 'child_id' column.
# Find which of them are themselves parents in FIMS.
# Their FIMS children become G3.
print('\n🔗 Chaining G2 → G3 via FIMS...')
g2_ids = set(df_nb01['child_id'].unique())
fims_g2_as_parent = fims[fims['fims_parent_id'].isin(g2_ids)].copy()
fims_g2_as_parent = fims_g2_as_parent.rename(columns={
    'fims_parent_id': 'g2_id',   # This is the G2 person (child in NB01)
    'fims_child_id': 'g3_id'     # This is the G3 person (grandchild of G1)
})
print(f'   G2 parents found in FIMS: {fims_g2_as_parent["g2_id"].nunique():,}')
print(f'   G3 children found: {fims_g2_as_parent["g3_id"].nunique():,}')

# ── Step 4: Get G3 outcome data from PSID extract ───────────────────────────
print('\n📂 Loading PSID extract for G3 outcomes...')
psid_raw = pd.read_csv(PSID_FILE, usecols=['ER30001', 'ER30002', 'ER30004', 'ER32000', 'ER32006', 'ER35152', 'ER82032'])
psid_raw['person_id'] = (psid_raw['ER30001'] * 1000) + psid_raw['ER30002']

# Filter to sample members
psid_g3 = psid_raw[psid_raw['ER32006'].isin([1, 2, 3, 4])].copy()

# Recode G3 variables (same rules as Notebook 01.2)
psid_g3['g3_educ_yrs'] = np.where(psid_g3['ER35152'].isin([0, 99]), np.nan, psid_g3['ER35152'])
psid_g3['g3_homeowner'] = np.where(psid_g3['ER82032'] == 1, 1,
                            np.where(psid_g3['ER82032'].isin([5, 8]), 0, np.nan))
psid_g3['g3_sex'] = psid_g3['ER32000']
psid_g3['birth_year_g3'] = np.where(psid_g3['ER30004'] > 0, 1968 - psid_g3['ER30004'], np.nan)

# Age restriction: G3 must be >= 25 in 2023
psid_g3 = psid_g3[psid_g3['birth_year_g3'] <= 1998].copy()

psid_g3 = psid_g3[['person_id', 'g3_educ_yrs', 'g3_homeowner', 'g3_sex', 'birth_year_g3']]
psid_g3 = psid_g3.rename(columns={'person_id': 'g3_id'})

# ── Step 5: Merge everything into a 3-generation dataset ─────────────────────
print('\n🔧 Building three-generation dataset...')

# Start: G2→G3 links merged with G3 outcomes
df_3gen = fims_g2_as_parent.merge(psid_g3, on='g3_id', how='inner')

# Merge G1 and G2 data from Notebook 01 output
# In NB01: parent_id = G1, child_id = G2
# race is already a string label ('White', 'Black', etc.) from NB01
g1g2_data = df_nb01[['parent_id', 'child_id', 'parent_homeowner', 'parent_educ_yrs',
                      'race', 'child_educ_yrs', 'child_homeowner_2023']].copy()
g1g2_data = g1g2_data.rename(columns={
    'parent_id': 'grandparent_id',
    'child_id': 'g2_id',
    'parent_homeowner': 'g1_homeowner',        # G1 homeownership (1968)
    'parent_educ_yrs': 'g1_educ_yrs',          # G1 education (years)
    'race': 'g1_race',                          # G1 race (string label from NB01)
    'child_educ_yrs': 'g2_educ_yrs',           # G2 education (years)
    'child_homeowner_2023': 'g2_homeowner',     # G2 homeownership (2023)
})

# Merge: bring G1+G2 data onto the G2→G3 frame
df_3gen = df_3gen.merge(g1g2_data, on='g2_id', how='inner')

# Rename g2_id to parent_id for the visualization code
df_3gen = df_3gen.rename(columns={'g2_id': 'parent_id'})

print(f'\n📈 Three-Generation Structure:')
print(f'   Generation 1 (Grandparents): {df_3gen["grandparent_id"].nunique():,} families')
print(f'   Generation 2 (Parents): {df_3gen["parent_id"].nunique():,} individuals')
print(f'   Generation 3 (Grandchildren): {len(df_3gen):,} individuals')
print(f'\n✅ Three-generation dataset built successfully from Notebook 01 output + FIMS!')

📂 Loading Notebook 01.2 output (G1–G2 pairs)...
   Loaded 3,560 rows
   Columns: ['child_id', 'parent_id', 'birth_year', 'child_sex', 'child_educ_yrs', 'child_homeowner_2023', 'parent_homeowner', 'parent_educ_yrs', 'parent_income_1968', 'parent_income_1968_log', 'race', 'state_1968', 'head_sex_1968', 'V103', 'V313', 'V81', 'V181', 'V93', 'V119', 'ER35152', 'ER82032', 'ER32000']

📂 Loading FIMS genealogy file...
   FIMS linkage pairs: 58,585

🔗 Chaining G2 → G3 via FIMS...
   G2 parents found in FIMS: 1,768
   G3 children found: 4,397

📂 Loading PSID extract for G3 outcomes...

🔧 Building three-generation dataset...

📈 Three-Generation Structure:
   Generation 1 (Grandparents): 39 families
   Generation 2 (Parents): 30 individuals
   Generation 3 (Grandchildren): 85 individuals

✅ Three-generation dataset built successfully from Notebook 01 output + FIMS!


## 2. Prepare Data for Visualization

Now we recode the numeric variables into human-readable labels for the tree visualizations. PSID stores race, sex, and education as numeric codes — we translate them here so the trees are easy to read.

In [ ]:
# ── Recode variables for display ─────────────────────────────────────────────
print('🔧 Preparing labels for visualization...')

df = df_3gen.copy()

# Only require grandparent columns to be complete
grandparent_cols = ['grandparent_id', 'g1_homeowner']
df_clean = df.dropna(subset=grandparent_cols)
print(f'After filtering for complete grandparent data: {len(df_clean):,} rows')

# ── Homeownership labels (all three generations) ────────────────────────────
homeowner_map = {1.0: 'Owner', 0.0: 'Renter'}
sex_map = {1.0: 'Male', 2.0: 'Female'}

df_clean['g1_homeowner_label'] = df_clean['g1_homeowner'].map(homeowner_map)
df_clean['g2_homeowner_label'] = df_clean['g2_homeowner'].map(homeowner_map)
df_clean['g3_homeowner_label'] = df_clean['g3_homeowner'].map(homeowner_map)

# ── Race label ───────────────────────────────────────────────────────────────
# g1_race is already a string ('White', 'Black', etc.) from Notebook 01.2
# No numeric-to-string conversion needed
df_clean['g1_race_label'] = df_clean['g1_race']

# ── Education: convert years to category codes for color mapping ─────────────
def years_to_ed_code(years):
    if pd.isna(years): return np.nan
    if years <= 5:   return 1   # 0-5 grades
    elif years <= 8: return 2   # 6-8 grades
    elif years <= 11: return 3  # 9-11 grades
    elif years == 12: return 4  # HS graduate
    elif years <= 14: return 5  # Some college
    elif years <= 16: return 6  # College+
    else: return 7              # Advanced degree

df_clean['g2_educ_code'] = df_clean['g2_educ_yrs'].apply(years_to_ed_code)

# ── G2 education labels ─────────────────────────────────────────────────────
parent_ed_map = {
    1: '0-5 grades', 2: '6-8 grades', 3: '9-11 grades',
    4: 'HS Graduate', 5: 'Some College', 6: 'College+',
    7: 'Advanced Degree'
}
df_clean['g2_ed_label'] = df_clean['g2_educ_code'].map(parent_ed_map)

# ── Sex labels ───────────────────────────────────────────────────────────────
df_clean['g3_sex_label'] = df_clean['g3_sex'].map(sex_map)

# ── G3 education category ───────────────────────────────────────────────────
def categorize_ed(years):
    if pd.isna(years): return 'Unknown'
    elif years < 12: return '<HS'
    elif years == 12: return 'HS'
    elif years < 16: return 'Some College'
    elif years >= 16: return 'College+'
    return 'Unknown'

df_clean['g3_ed_category'] = df_clean['g3_educ_yrs'].apply(categorize_ed)

# ── Family IDs ───────────────────────────────────────────────────────────────
unique_grandparents = df_clean['grandparent_id'].unique()
family_id_map = {gid: f'Family_{i:05d}' for i, gid in enumerate(unique_grandparents, 1)}
df_clean['family_id'] = df_clean['grandparent_id'].map(family_id_map)

# ── Summary ──────────────────────────────────────────────────────────────────
g1_counts = df_clean['grandparent_id'].nunique()
g2_counts = df_clean['parent_id'].nunique()
g3_counts = len(df_clean)

print(f'\n📈 Three-Generation Structure:')
print(f'   Generation 1 (Grandparents): {g1_counts:,} families')
print(f'   Generation 2 (Parents): {g2_counts:,} families')
print(f'   Generation 3 (Grandchildren): {g3_counts:,} individuals')

if 'g1_race_label' in df_clean.columns:
    print(f'\n📊 Race distribution (G1):')
    race_dist = df_clean.groupby('g1_race_label')['grandparent_id'].nunique()
    for race, count in race_dist.items():
        print(f'   {race}: {count:,} families')

print(f'\n📊 Homeownership labels check:')
print(f'   G1: {df_clean["g1_homeowner_label"].value_counts().to_dict()}')
print(f'   G2: {df_clean["g2_homeowner_label"].value_counts(dropna=False).to_dict()}')
print(f'   G3: {df_clean["g3_homeowner_label"].value_counts(dropna=False).to_dict()}')

print(f'\n✅ Data preparation complete!')

🔧 Preparing labels for visualization...
After filtering for complete grandparent data: 85 rows

📈 Three-Generation Structure:
   Generation 1 (Grandparents): 39 families
   Generation 2 (Parents): 30 families
   Generation 3 (Grandchildren): 85 individuals

📊 Race distribution (G1):
   Black: 30 families
   White: 9 families

📊 Homeownership labels check:
   G1: {'Renter': 43, 'Owner': 42}
   G2: {'Owner': 45, 'Renter': 40}
   G3: {nan: 41, 'Owner': 27, 'Renter': 17}

✅ Data preparation complete!


## 3. Build and Visualize Family Trees

The tree builder takes all rows for a given family (identified by `grandparent_id`) and reconstructs the three-generation structure:

- **G1 node**: The 1968 household head. Labeled with race and homeownership status.
- **G2 nodes**: Their children (one per unique `parent_id`). Labeled with education category.
- **G3 nodes**: Grandchildren. Labeled with sex and years of education.

The visualization uses **color** to show education level (red = less than HS, green = college, blue = graduate+) and **shape** to show homeownership (circle = owner, square = renter, diamond = unknown).

In [ ]:
# Build 3-generation tree
def build_three_gen_tree(family_rows):
    """Build a 3-generation family tree showing educational outcomes.

    Research Question: Does Gen1 homeownership affect Gen2 & Gen3 education?

    Gen 1 = Grandparent (1968 homeowner status)
    Gen 2 = Parent (has g2_educ_yrs measured as adult)
    Gen 3 = Grandchild (has g3_educ_yrs measured as adult)
    """
    first_row = family_rows.iloc[0]

    tree = {
        'family_id': first_row['family_id'],
        'g1': {
            'id': first_row['grandparent_id'],
            'race': first_row.get('g1_race_label', 'Unknown'),
            'homeowner_1968': first_row.get('g1_homeowner_label', 'Unknown'),
        },
        'g2': {},  # Will be dict of parent_id -> parent data
        'g3': []   # List of all grandchildren
    }

    # Group by parent_id to get Generation 2 structure
    for parent_id in family_rows['parent_id'].unique():
        parent_rows = family_rows[family_rows['parent_id'] == parent_id]
        first_parent_row = parent_rows.iloc[0]

        tree['g2'][parent_id] = {
            'id': parent_id,
            'education_category': first_parent_row.get('g2_ed_label', 'Unknown'),
            'education_code': first_parent_row.get('g2_educ_code'),
            'education_years': first_parent_row.get('g2_educ_yrs'),
            'homeowner': first_parent_row.get('g2_homeowner_label', 'Unknown'),
            'children': []
        }

        # Gen 3 - Add each grandchild with their education
        for idx, row in parent_rows.iterrows():
            child_data = {
                'parent_id': parent_id,
                'sex': row.get('g3_sex_label', 'Unknown'),
                'education_years': row.get('g3_educ_yrs'),
                'education_category': row.get('g3_ed_category', 'Unknown'),
                'homeowner': row.get('g3_homeowner_label', 'Unknown'),
            }
            tree['g2'][parent_id]['children'].append(child_data)
            tree['g3'].append(child_data)

    return tree

print("✅ Three-generation tree builder defined")

✅ Three-generation tree builder defined


In [ ]:
# Visualize 3-generation tree with CLEAR education labels
def visualize_three_gen_tree(tree, figsize=(18, 12)):
    """Visualize educational outcomes across 3 generations.
    COLOR = Education level, SHAPE = Homeownership
    """
    G = nx.DiGraph()

    def get_education_color_from_years(years):
        """Color based on years of education."""
        if pd.isna(years):
            return '#90A4AE'  # Gray
        if years < 12:
            return '#D32F2F'  # Red
        elif years == 12:
            return '#F57C00'  # Orange
        elif years < 16:
            return '#FBC02D'  # Yellow
        elif years == 16:
            return '#388E3C'  # Green
        else:
            return '#1976D2'  # Blue

    def get_education_color_from_code(code):
        """Color based on education category codes (1-7)."""
        if pd.isna(code):
            return '#90A4AE'
        code = int(code)
        if code <= 3:  # 0-11 grades
            return '#D32F2F'
        elif code == 4:  # HS
            return '#F57C00'
        elif code == 5:  # Some College
            return '#FBC02D'
        elif code == 6:  # College
            return '#388E3C'
        elif code >= 7:  # Advanced
            return '#1976D2'
        return '#90A4AE'

    def get_ed_label_from_code(code):
        """Get SHORT clear label from code."""
        if pd.isna(code):
            return '?'
        code = int(code)
        if code <= 3:
            return '<HS'
        elif code == 4:
            return 'HS'
        elif code == 5:
            return 'SomeColl'
        elif code == 6:
            return 'College'
        elif code >= 7:
            return 'Grad+'
        return '?'

    def get_ed_label_from_years(years):
        """Get SHORT clear label from years."""
        if pd.isna(years):
            return '?'
        y = int(years)
        if y < 12:
            return f'{y}y (<HS)'
        elif y == 12:
            return '12y (HS)'
        elif y < 16:
            return f'{y}y (SomeColl)'
        elif y == 16:
            return '16y (College)'
        else:
            return f'{y}y (Grad+)'

    def get_ownership_shape(owner_status):
        if owner_status == 'Owner':
            return 'o'
        elif owner_status == 'Renter':
            return 's'
        else:
            return 'd'

    nodes_by_shape = {'o': [], 's': [], 'd': []}
    node_colors = {}
    node_labels = {}

    # Gen 1
    grandparent_node = 'g1'
    homeowner_status = tree['g1']['homeowner_1968']
    g1_color = '#7E57C2'
    g1_shape = get_ownership_shape(homeowner_status)
    g1_label = f"G1\n{tree['g1']['race']}\n{homeowner_status}"

    G.add_node(grandparent_node, label=g1_label, generation=1)
    nodes_by_shape[g1_shape].append(grandparent_node)
    node_colors[grandparent_node] = g1_color
    node_labels[grandparent_node] = g1_label

    # Gen 2
    parent_nodes = []
    for i, (parent_id, parent_data) in enumerate(tree['g2'].items()):
        parent_node = f"g2_{i}"
        parent_nodes.append(parent_node)

        parent_ed_code = parent_data.get('education_code')
        parent_owner = parent_data.get('homeowner', 'Unknown')

        parent_color = get_education_color_from_code(parent_ed_code)
        parent_shape = get_ownership_shape(parent_owner)

        # CLEAR label
        parent_label = f"G2-{i+1}\n{get_ed_label_from_code(parent_ed_code)}"

        G.add_node(parent_node, label=parent_label, generation=2)
        G.add_edge(grandparent_node, parent_node)

        nodes_by_shape[parent_shape].append(parent_node)
        node_colors[parent_node] = parent_color
        node_labels[parent_node] = parent_label

        # Gen 3
        for j, child in enumerate(parent_data['children']):
            child_node = f"g3_{i}_{j}"

            ed_years = child['education_years']
            child_color = get_education_color_from_years(ed_years)
            child_shape = get_ownership_shape(child.get('homeowner', 'Unknown'))

            # CLEAR label
            child_label = f"G3\n{child['sex']}\n{get_ed_label_from_years(ed_years)}"

            G.add_node(child_node, label=child_label, generation=3)
            G.add_edge(parent_node, child_node)

            nodes_by_shape[child_shape].append(child_node)
            node_colors[child_node] = child_color
            node_labels[child_node] = child_label

    # Layout
    pos = {}
    pos[grandparent_node] = (0, 2)

    num_parents = len(parent_nodes)
    if num_parents == 1:
        parent_x_positions = [0]
    else:
        parent_x_positions = np.linspace(-2, 2, num_parents)

    for i, parent_node in enumerate(parent_nodes):
        pos[parent_node] = (parent_x_positions[i], 1)

    for i, (parent_id, parent_data) in enumerate(tree['g2'].items()):
        num_children = len(parent_data['children'])
        parent_x = parent_x_positions[i]

        if num_children == 1:
            child_x_positions = [parent_x]
        else:
            spread = min(1.3, num_children * 0.4)
            child_x_positions = np.linspace(parent_x - spread/2, parent_x + spread/2, num_children)

        for j in range(num_children):
            child_node = f"g3_{i}_{j}"
            pos[child_node] = (child_x_positions[j], 0)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#666666',
                          width=1.5, alpha=0.4, arrows=False)

    for shape, node_list in nodes_by_shape.items():
        if node_list:
            colors = [node_colors[n] for n in node_list]
            nx.draw_networkx_nodes(G, pos, nodelist=node_list,
                                  node_color=colors, node_size=5250,
                                  node_shape=shape, ax=ax,
                                  edgecolors='black', linewidths=2)

    nx.draw_networkx_labels(G, pos, node_labels, font_size=9,
                           font_color='white', font_weight='bold',
                           verticalalignment='center', ax=ax)

    # Title
    num_g2 = len(parent_nodes)
    num_g3 = len(tree['g3'])
    title = f"Family: {tree['family_id']}\n"
    title += f"G1 (1968): {tree['g1']['race']} {homeowner_status}"
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

    # Legend
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    ed_legend = [
        Patch(facecolor='#D32F2F', label='<12y = <HS'),
        Patch(facecolor='#F57C00', label='12y = HS'),
        Patch(facecolor='#FBC02D', label='13-15y = Some College'),
        Patch(facecolor='#388E3C', label='16y = College'),
        Patch(facecolor='#1976D2', label='17+y = Graduate+'),
    ]

    shape_legend = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
               markersize=10, label='Owner', markeredgecolor='black'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='gray',
               markersize=10, label='Renter', markeredgecolor='black'),
        Line2D([0], [0], marker='d', color='w', markerfacecolor='gray',
               markersize=10, label='Unknown', markeredgecolor='black'),
    ]

    first_legend = ax.legend(handles=ed_legend, loc='upper left',
                            title='Years of Education', fontsize=8)
    ax.add_artist(first_legend)
    ax.legend(handles=shape_legend, loc='upper right',
             title='Homeownership', fontsize=8)

    ax.axis('off')
    plt.tight_layout()
    plt.show()

print("✅ Visualization function defined")

✅ Visualization function defined


In [ ]:
# Interactive widgets
print("🎨 Setting up interface...")

race_widget = widgets.Dropdown(
    options=['All', 'White', 'Black', 'Hispanic', 'Other'],
    value='All',
    description='Gen1 Race:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='350px')
)

homeowner_widget = widgets.Dropdown(
    options=['All', 'Owner', 'Renter'],
    value='All',
    description='Gen1 Homeowner (1968):',
    style={'description_width': '200px'},
    layout=widgets.Layout(width='400px')
)

search_widget = widgets.Text(
    placeholder='Enter Family ID (e.g., Family_00001)',
    description='Family ID:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='350px')
)

random_button = widgets.Button(
    description='🎲 Show Random Family',
    button_style='success',
    layout=widgets.Layout(width='300px', height='45px')
)

search_button = widgets.Button(
    description='🔍 Search by ID',
    button_style='info',
    layout=widgets.Layout(width='200px', height='45px')
)

output = Output()

print("✅ Widgets created!")

🎨 Setting up interface...
✅ Widgets created!


In [ ]:
# Functions
def show_random_family(b=None):
    with output:
        clear_output(wait=True)

        filtered = df_clean.copy()

        # Apply race filter
        if race_widget.value != 'All':
            if 'g1_race_label' in filtered.columns:
                filtered = filtered[filtered['g1_race_label'] == race_widget.value]
                print(f"🔍 Filtered by race ({race_widget.value}): {filtered['grandparent_id'].nunique()} families")

        # Apply homeownership filter
        if homeowner_widget.value != 'All':
            filtered = filtered[filtered['g1_homeowner_label'] == homeowner_widget.value]
            print(f"🔍 Filtered by homeowner ({homeowner_widget.value}): {filtered['grandparent_id'].nunique()} families")

        if len(filtered) == 0:
            print("❌ No families found matching your criteria!")
            print("   Try relaxing some filters.")
            return

        # Pick random grandparent
        family_ids = filtered['family_id'].unique()
        random_family_id = np.random.choice(family_ids)

        family_rows = df_clean[df_clean['family_id'] == random_family_id]

        print(f"\n📊 Showing: {random_family_id}")
        print(f"   {len(family_rows)} total descendants in dataset")
        print(f"   {family_rows['parent_id'].nunique()} Gen2 parents")
        print("=" * 70)

        tree = build_three_gen_tree(family_rows)
        visualize_three_gen_tree(tree)

def search_by_id(b=None):
    with output:
        clear_output(wait=True)

        search_id = search_widget.value.strip()
        if not search_id:
            print("❌ Please enter a Family ID")
            return

        family_rows = df_clean[df_clean['family_id'] == search_id]

        if len(family_rows) == 0:
            print(f"❌ Family ID '{search_id}' not found")
            return

        print(f"\n📊 Showing: {search_id}")
        print("=" * 70)

        tree = build_three_gen_tree(family_rows)
        visualize_three_gen_tree(tree)

random_button.on_click(show_random_family)
search_button.on_click(search_by_id)

print("✅ Functions connected!")

✅ Functions connected!


In [ ]:
# Display interface
print("\n" + "="*80)
print("🌳 INTERGENERATIONAL EDUCATION EXPLORER (Longitudinal)")
print("="*80)
print("\nResearch Question: Does 1968 homeownership affect educational")
print("outcomes for children and grandchildren measured decades later?")
print("\nNote: Gen2 & Gen3 education measured when they were ADULTS.")
print("This shows COMPLETED education, not childhood education.\n")

filter_box = VBox([
    widgets.HTML("<h3>Filter by Gen1 (1968 Household)</h3>"),
    race_widget,
    homeowner_widget,
    random_button
])

search_box = VBox([
    widgets.HTML("<h3>Search Specific Family</h3>"),
    search_widget,
    search_button
])

top_section = HBox([filter_box, search_box])

display(top_section)
display(output)

print("\n✅ Filter by 1968 homeownership, see adult outcomes decades later!")


🌳 INTERGENERATIONAL EDUCATION EXPLORER (Longitudinal)

Research Question: Does 1968 homeownership affect educational
outcomes for children and grandchildren measured decades later?

Note: Gen2 & Gen3 education measured when they were ADULTS.
This shows COMPLETED education, not childhood education.



Output()


✅ Filter by 1968 homeownership, see adult outcomes decades later!
